# LeetCode #162 — Find Peak Element: tìm nhị phân theo "hướng dốc lên"

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search

Skeleton trên LeetCode:

```python
class Solution:
    def findPeakElement(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Hiểu **đỉnh** (peak): phần tử lớn hơn **cả hai** hàng xóm, với quy ước `nums[-1] = nums[n] = −∞`.
2. Vì sao **luôn tồn tại** ít nhất một đỉnh.
3. Tìm nhị phân **trên mảng chưa sắp xếp**: so `nums[mid]` với `nums[mid + 1]` rồi đi về phía **dốc lên**.
4. Mẫu `while lo < hi` với `hi = mid` / `lo = mid + 1`.

# 1. Bài toán là gì?

Một **phần tử đỉnh** là phần tử **lớn hơn hẳn** các hàng xóm của nó.

Cho mảng `nums` (đánh số từ 0), tìm một phần tử đỉnh và trả về **chỉ số** của nó. Nếu có nhiều đỉnh, trả về **bất kỳ** đỉnh nào.

Quy ước `nums[-1] = nums[n] = −∞`: một phần tử luôn lớn hơn phần tử "ở ngoài mảng".

Thuật toán phải chạy trong `O(log n)`.

**Constraints (theo đề):**

- `1 <= nums.length <= 1000`
- `-2^31 <= nums[i] <= 2^31 - 1`
- `nums[i] != nums[i + 1]` với mọi `i` hợp lệ (hai phần tử **kề nhau** luôn khác nhau).

---

## Ví dụ

```text
[1, 2, 3, 1]            ->  2        (nums[2] = 3)
[1, 2, 1, 3, 5, 6, 4]   ->  1 hoặc 5 (nums[1] = 2, nums[5] = 6 đều là đỉnh)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Áp dụng vào bài này

Mảng **không** được sắp xếp, nhưng vẫn **tìm nhị phân** được vì ta không tìm một giá trị mà tìm một vị trí thỏa điều kiện "đi về phía dốc lên thì chắc chắn gặp đỉnh". So `nums[mid]` với `nums[mid + 1]`: nếu đang dốc lên thì có đỉnh ở nửa phải (`lo = mid + 1`), ngược lại có đỉnh ở `[lo, mid]` (`hi = mid`). Đây chính là khuôn "tìm vị trí đầu tiên thỏa điều kiện" với `lo < hi` và `hi = mid` ở trên.

# 3. Nền tảng

In [4]:
from typing import List
import random, time

def is_peak(nums: List[int], i: int) -> bool:
    left = nums[i - 1] if i > 0 else float("-inf")
    right = nums[i + 1] if i + 1 < len(nums) else float("-inf")
    return nums[i] > left and nums[i] > right

a = [1, 2, 1, 3, 5, 6, 4]
print("các đỉnh của", a, ":", [i for i in range(len(a)) if is_peak(a, i)])
print("mảng một phần tử [7]: đỉnh?", is_peak([7], 0))

các đỉnh của [1, 2, 1, 3, 5, 6, 4] : [1, 5]
mảng một phần tử [7]: đỉnh? True


`float("-inf")` nhỏ hơn mọi số nguyên — đúng quy ước `−∞` ở hai đầu.

# 4. Baseline: duyệt tuyến tính

Đi từ trái sang: phần tử đầu tiên mà `nums[i] > nums[i + 1]` là một đỉnh (vì trước đó dãy đang tăng nên `nums[i] > nums[i − 1]`). Nếu không có, phần tử cuối là đỉnh. `O(n)` — chưa đạt yêu cầu `O(log n)`.

In [5]:
def find_peak_linear(nums: List[int]) -> int:
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            return i
    return len(nums) - 1

print(find_peak_linear([1, 2, 3, 1]), find_peak_linear([1, 2, 1, 3, 5, 6, 4]), find_peak_linear([1, 2, 3]))

2 1 2


# 5. Ý tưởng chính: đi về phía dốc lên

Đứng ở `mid`, so với hàng xóm phải:

- `nums[mid] < nums[mid + 1]`: đang **leo dốc lên** sang phải. Đi tiếp về bên phải, dãy hoặc tăng mãi tới cuối (phần tử cuối là đỉnh vì `nums[n] = −∞`), hoặc đến lúc giảm — chỗ bắt đầu giảm là đỉnh. Vậy **bên phải `mid` chắc chắn có đỉnh** → `lo = mid + 1`.
- `nums[mid] > nums[mid + 1]`: dốc lên về bên **trái** (nhìn từ `mid + 1`). Lập luận tương tự: đoạn `[lo, mid]` chắc chắn có đỉnh (có thể chính là `mid`) → `hi = mid`.

```text
     ●
   ●   ●           mid ở sườn trái (nums[mid] < nums[mid+1]) → đỉnh ở bên phải
 ●       ●
```

Không cần mảng sắp xếp — chỉ cần **biết nửa nào chắc chắn chứa đáp án**.

```text
lo, hi = 0, n - 1
while lo < hi:
    mid = (lo + hi) // 2
    if nums[mid] < nums[mid + 1]: lo = mid + 1
    else:                         hi = mid
return lo
```

`mid < hi` nên `mid + 1` luôn hợp lệ.

In [6]:
def find_peak(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

for t in [[1, 2, 3, 1], [1, 2, 1, 3, 5, 6, 4], [5], [3, 2, 1], [1, 2, 3]]:
    i = find_peak(t)
    print(t, "->", i, "| là đỉnh?", is_peak(t, i))

[1, 2, 3, 1] -> 2 | là đỉnh? True
[1, 2, 1, 3, 5, 6, 4] -> 5 | là đỉnh? True
[5] -> 0 | là đỉnh? True
[3, 2, 1] -> 0 | là đỉnh? True
[1, 2, 3] -> 2 | là đỉnh? True


# 6. Mô phỏng

In [7]:
def trace_peak(nums):
    lo, hi, step = 0, len(nums) - 1, 0
    while lo < hi:
        mid = (lo + hi) // 2
        step += 1
        if nums[mid] < nums[mid + 1]:
            note = f"{nums[mid]} < {nums[mid + 1]} → dốc lên bên phải → lo = {mid + 1}"
            lo = mid + 1
        else:
            note = f"{nums[mid]} > {nums[mid + 1]} → đỉnh ở [lo, mid] → hi = {mid}"
            hi = mid
        print(f"bước {step}: mid = {mid} | {note:<45} | khoảng {nums[lo:hi + 1]}")
    print(f"→ chỉ số {lo}, giá trị {nums[lo]}")
    return lo

_ = trace_peak([1, 2, 1, 3, 5, 6, 4])

bước 1: mid = 3 | 3 < 5 → dốc lên bên phải → lo = 4             | khoảng [5, 6, 4]
bước 2: mid = 5 | 6 > 4 → đỉnh ở [lo, mid] → hi = 5             | khoảng [5, 6]
bước 3: mid = 4 | 5 < 6 → dốc lên bên phải → lo = 5             | khoảng [6]
→ chỉ số 5, giá trị 6


Thuật toán tìm ra đỉnh ở chỉ số `5`; đỉnh ở chỉ số `1` cũng đúng nhưng nằm ở nửa bị loại — đề cho phép trả **bất kỳ** đỉnh nào.

# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def findPeakElement(self, nums: List[int]) -> int:
        lo, hi = 0, len(nums) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] < nums[mid + 1]:
                lo = mid + 1
            else:
                hi = mid
        return lo

Vì có thể có nhiều đáp án, ta **không** so với một giá trị cố định mà kiểm tra chỉ số trả về **là đỉnh**.

In [9]:
def check(name, nums):
    i = Solution().findPeakElement(nums)
    assert 0 <= i < len(nums) and is_peak(nums, i), (name, nums, i)
    print(f"{name:<14} PASS  chỉ số {i} (giá trị {nums[i]})")


check("Ví dụ 1", [1, 2, 3, 1])
check("Ví dụ 2", [1, 2, 1, 3, 5, 6, 4])
check("Một phần tử", [-2 ** 31])
check("Tăng dần", [1, 2, 3, 4])
check("Giảm dần", [4, 3, 2, 1])

def random_no_equal_neighbors(n):
    a = [random.randint(-5, 5)]
    while len(a) < n:
        x = random.randint(-5, 5)
        if x != a[-1]:
            a.append(x)
    return a

random.seed(162)
for _ in range(5000):
    nums = random_no_equal_neighbors(random.randint(1, 15))
    i = Solution().findPeakElement(nums)
    assert is_peak(nums, i), nums
print("5000 mảng ngẫu nhiên: chỉ số trả về luôn là đỉnh")

class CountReads(list):
    reads = 0
    def __getitem__(self, i):
        CountReads.reads += 1
        return super().__getitem__(i)

CountReads.reads = 0
Solution().findPeakElement(CountReads(range(1000)))
print("n = 1000, mảng tăng dần: số lần đọc phần tử =", CountReads.reads, "(tuyến tính cần ~1000)")

Ví dụ 1        PASS  chỉ số 2 (giá trị 3)
Ví dụ 2        PASS  chỉ số 5 (giá trị 6)
Một phần tử    PASS  chỉ số 0 (giá trị -2147483648)
Tăng dần       PASS  chỉ số 3 (giá trị 4)
Giảm dần       PASS  chỉ số 0 (giá trị 4)
5000 mảng ngẫu nhiên: chỉ số trả về luôn là đỉnh
n = 1000, mảng tăng dần: số lần đọc phần tử = 18 (tuyến tính cần ~1000)


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** đoạn `[lo, hi]` luôn chứa ít nhất một đỉnh **của mảng gốc**, và `nums[lo − 1] < nums[lo]` (hoặc `lo = 0`), `nums[hi] > nums[hi + 1]` (hoặc `hi = n − 1`).

- Ban đầu đúng nhờ quy ước `−∞` ở hai đầu.
- `nums[mid] < nums[mid + 1]` → đặt `lo = mid + 1` giữ điều kiện "bên trái `lo` thấp hơn `lo`".
- Ngược lại → đặt `hi = mid` giữ điều kiện "bên phải `hi` thấp hơn `hi`".
- Khi `lo == hi`: phần tử đó lớn hơn cả hai hàng xóm → là đỉnh.

**Vì sao luôn có đỉnh?** Phần tử lớn nhất của mảng lớn hơn mọi hàng xóm (các phần tử kề khác nhau, hai đầu là `−∞`).

```text
Time:  O(log n)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên quy ước `−∞` ở hai đầu (chỉ xét các phần tử ở giữa)

In [10]:
def wrong_interior_only(nums):
    for i in range(1, len(nums) - 1):              # SAI: bỏ qua nums[0] và nums[n-1]
        if nums[i] > nums[i - 1] and nums[i] > nums[i + 1]:
            return i
    return -1

for t in [[1, 2, 3], [3, 2, 1], [5]]:
    print(t, "-> sai:", wrong_interior_only(t), "| đúng:", find_peak(t))

[1, 2, 3] -> sai: -1 | đúng: 2
[3, 2, 1] -> sai: -1 | đúng: 0
[5] -> sai: -1 | đúng: 0


Mảng tăng dần có đỉnh ở **cuối**, mảng giảm dần có đỉnh ở **đầu**, mảng một phần tử thì chính nó là đỉnh — cả ba đều bị bỏ sót.

Một biến thể của lỗi này: viết `nums[i - 1]` với `i = 0`. Python **không** báo lỗi mà lấy `nums[-1]` = phần tử **cuối** — so sánh sai một cách âm thầm. So với `nums[mid + 1]` (luôn hợp lệ vì `mid < hi`) thì không gặp vấn đề này.

## Lỗi 2: `hi = mid − 1` khi `nums[mid] > nums[mid + 1]`

In [11]:
def wrong_hi_minus(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid - 1                    # SAI: mid có thể chính là đỉnh duy nhất trong đoạn
    return lo

for t in [[1, 3, 2, 1], [1, 2, 3, 1], [2, 1]]:
    r = wrong_hi_minus(t)
    print(t, "-> sai:", r, "đỉnh?", is_peak(t, r), "| đúng:", find_peak(t))

[1, 3, 2, 1] -> sai: 0 đỉnh? False | đúng: 1
[1, 2, 3, 1] -> sai: 2 đỉnh? True | đúng: 2
[2, 1] -> sai: 0 đỉnh? True | đúng: 0


## Lỗi 3: Dùng `while lo <= hi` với `hi = mid`

Khi `lo == hi`, `mid == hi` và `nums[mid + 1]` có thể **vượt mảng** (`IndexError`), hoặc vòng lặp không thu hẹp được. Mẫu "tìm vị trí" dùng `lo < hi`.

# 10. Mẹo nhận diện pattern

- **`O(log n)` nhưng mảng không sắp xếp** → tìm điều kiện cho biết **nửa nào chắc chắn chứa đáp án** (ở đây: hướng dốc lên).
- **"Trả về bất kỳ"** → không cần tìm tất cả; chỉ cần đảm bảo tồn tại.

Bài liên quan: **#852 Peak Index in a Mountain Array**, **#1095 Find in Mountain Array** (Hard), **#153 Find Minimum in Rotated Sorted Array**, **#1901 Find a Peak Element II** (lưới).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy thuật toán trên `[1, 5, 2, 8, 9, 7, 3]`. Đỉnh nào được trả về? Còn đỉnh nào khác?

## Bài 2 — Điền chỗ trống

In [12]:
def exercise_2_find_peak(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO: so nums[mid] với nums[mid + 1]
        pass
    return lo


# assert is_peak([1, 2, 3, 1], exercise_2_find_peak([1, 2, 3, 1]))
# assert is_peak([1, 2, 1, 3, 5, 6, 4], exercise_2_find_peak([1, 2, 1, 3, 5, 6, 4]))
# assert exercise_2_find_peak([9]) == 0

## Bài 3 — Biến thể: Peak Index in a Mountain Array (LeetCode #852)

Mảng "núi": tăng ngặt rồi giảm ngặt (đỉnh duy nhất, không ở hai đầu). Trả về chỉ số đỉnh trong `O(log n)`.

In [13]:
def exercise_3_peak_mountain(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_peak_mountain([0, 1, 0]) == 1
# assert exercise_3_peak_mountain([0, 2, 1, 0]) == 1
# assert exercise_3_peak_mountain([0, 10, 5, 2]) == 1
# assert exercise_3_peak_mountain([3, 4, 5, 1]) == 2

## Bài 4 — Biến thể: tìm **thung lũng** (phần tử nhỏ hơn cả hai hàng xóm, hai đầu là `+∞`)

In [14]:
def exercise_4_find_valley(nums: List[int]) -> int:
    # TODO
    pass


def is_valley(nums, i):
    left = nums[i - 1] if i > 0 else float("inf")
    right = nums[i + 1] if i + 1 < len(nums) else float("inf")
    return nums[i] < left and nums[i] < right

# assert is_valley([3, 1, 2], exercise_4_find_valley([3, 1, 2]))
# assert is_valley([5, 4, 3, 2], exercise_4_find_valley([5, 4, 3, 2]))
# assert exercise_4_find_valley([7]) == 0

## Bài 5 — Mini challenge: Find a Peak Element II (LeetCode #1901)

Lưới `m × n`, hai ô kề nhau (4 hướng) luôn khác nhau, ngoài biên là `−1`. Tìm một ô lớn hơn **cả 4** ô kề trong `O(m log n)` hoặc `O(n log m)`.

Gợi ý: tìm nhị phân trên **cột**; ở cột `mid`, lấy hàng `r` có giá trị **lớn nhất**; so `grid[r][mid]` với `grid[r][mid + 1]`.

In [15]:
def exercise_5_find_peak_grid(mat: List[List[int]]) -> List[int]:
    # TODO
    pass


def is_peak_2d(mat, r, c):
    m, n = len(mat), len(mat[0])
    v = mat[r][c]
    for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        rr, cc = r + dr, c + dc
        if 0 <= rr < m and 0 <= cc < n and mat[rr][cc] >= v:
            return False
    return True

# g5 = [[1, 4], [3, 2]]; r5 = exercise_5_find_peak_grid(g5); assert is_peak_2d(g5, r5[0], r5[1])
# g5b = [[10, 20, 15], [21, 30, 14], [7, 16, 32]]; r5b = exercise_5_find_peak_grid(g5b); assert is_peak_2d(g5b, r5b[0], r5b[1])

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_peak` sau khi tự làm.

**Bài 3**
- Mức 1: đúng là thuật toán bài gốc (mảng núi là trường hợp riêng).

**Bài 4**
- Mức 1: đảo chiều so sánh: `nums[mid] > nums[mid + 1]` → dốc **xuống** bên phải → `lo = mid + 1`.

**Bài 5**
- Mức 1: ô lớn nhất trong cột `mid` đã lớn hơn ô trên và dưới.
- Mức 2: nếu nó nhỏ hơn ô bên phải → bên phải chắc chắn có đỉnh (lập luận "đi lên"), `lo = mid + 1`; ngược lại `hi = mid`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [16]:
arr1 = [1, 5, 2, 8, 9, 7, 3]
_ = trace_peak(arr1)
print("mọi đỉnh:", [i for i in range(len(arr1)) if is_peak(arr1, i)])

bước 1: mid = 3 | 8 < 9 → dốc lên bên phải → lo = 4             | khoảng [9, 7, 3]
bước 2: mid = 5 | 7 > 3 → đỉnh ở [lo, mid] → hi = 5             | khoảng [9, 7]
bước 3: mid = 4 | 9 > 7 → đỉnh ở [lo, mid] → hi = 4             | khoảng [9]
→ chỉ số 4, giá trị 9
mọi đỉnh: [1, 4]


## Đáp án Bài 2

In [17]:
def answer_2_find_peak(nums: List[int]) -> int:
    return Solution().findPeakElement(nums)

assert is_peak([1, 2, 3, 1], answer_2_find_peak([1, 2, 3, 1]))
assert is_peak([1, 2, 1, 3, 5, 6, 4], answer_2_find_peak([1, 2, 1, 3, 5, 6, 4]))
assert answer_2_find_peak([9]) == 0
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [18]:
def answer_3_peak_mountain(arr: List[int]) -> int:
    lo, hi = 0, len(arr) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < arr[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert answer_3_peak_mountain([0, 1, 0]) == 1
assert answer_3_peak_mountain([0, 2, 1, 0]) == 1
assert answer_3_peak_mountain([0, 10, 5, 2]) == 1
assert answer_3_peak_mountain([3, 4, 5, 1]) == 2
for _ in range(2000):
    p = random.randint(1, 8); q = random.randint(1, 8)
    up = sorted(random.sample(range(10, 100), p)); peak = up[-1] + 1 + random.randint(0, 5)
    down = sorted(random.sample(range(peak), q), reverse=True)
    arr = up + [peak] + down
    assert answer_3_peak_mountain(arr) == arr.index(max(arr))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [19]:
def answer_4_find_valley(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert is_valley([3, 1, 2], answer_4_find_valley([3, 1, 2]))
assert is_valley([5, 4, 3, 2], answer_4_find_valley([5, 4, 3, 2]))
assert answer_4_find_valley([7]) == 0
for _ in range(3000):
    nums = random_no_equal_neighbors(random.randint(1, 12))
    assert is_valley(nums, answer_4_find_valley(nums))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [20]:
def answer_5_find_peak_grid(mat: List[List[int]]) -> List[int]:
    m, n = len(mat), len(mat[0])
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        r = max(range(m), key=lambda i: mat[i][mid])
        if mat[r][mid] < mat[r][mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    r = max(range(m), key=lambda i: mat[i][lo])
    return [r, lo]

g5 = [[1, 4], [3, 2]]; r5 = answer_5_find_peak_grid(g5); assert is_peak_2d(g5, r5[0], r5[1])
g5b = [[10, 20, 15], [21, 30, 14], [7, 16, 32]]; r5b = answer_5_find_peak_grid(g5b); assert is_peak_2d(g5b, r5b[0], r5b[1])
for _ in range(1000):
    m, n = random.randint(1, 5), random.randint(1, 5)
    vals = random.sample(range(1, 100), m * n)       # mọi ô khác nhau → hai ô kề luôn khác nhau
    mat = [vals[i * n:(i + 1) * n] for i in range(m)]
    r, c = answer_5_find_peak_grid(mat)
    assert is_peak_2d(mat, r, c), mat
print("Bài 5 PASS")

Bài 5 PASS

Vì sao đúng? Ô `(r, mid)` là lớn nhất cột `mid`. Nếu nó nhỏ hơn `(r, mid+1)` thì ô lớn nhất của cột `mid+1` còn lớn hơn mọi ô của cột `mid` → "đi lên" sang phải luôn được, nên nửa phải chứa đỉnh — giống bài gốc nhưng mỗi "phần tử" là cả một cột.

# 14. Tổng kết

> Không cần sắp xếp để tìm nhị phân — chỉ cần biết nửa nào chắc chắn có đáp án. So `nums[mid]` với `nums[mid + 1]`: nhỏ hơn thì đỉnh ở bên phải (`lo = mid + 1`), lớn hơn thì đỉnh ở `[lo, mid]` (`hi = mid`). `O(log n)`.

---

## Checklist trước khi nộp

- [ ] So với `nums[mid + 1]` (luôn hợp lệ vì `mid < hi`)?
- [ ] `hi = mid` (không `mid − 1`)?
- [ ] `while lo < hi`?
- [ ] Mảng một phần tử trả `0`?